# အတန်း 04 - စက်နည်းအသုံးပြုမှု ဒီဇိုင်းပုံစံ

ဤအတန်းတွင် သင်သည် Microsoft Agent Framework (Python) ကို အသုံးပြု၍ AI ကိုယ်စားလှယ်များအတွက် **စက်နည်းအသုံးပြုမှု** ဒီဇိုင်းပုံစံကို သင်ယူမည်ဖြစ်သည်။ ကျွန်ုပ်တို့သည် အောက်ပါအချက်များကို ဖုံးကွယ်ပါသည်။

- `@tool` decorator နှင့် typed ပါရာမီတာများဖြင့် function tools သတ်မှတ်ခြင်း
- မော်ဒယ်သည် tools တစ်ခုချင်းစီလုပ်ဆောင်သောအရာများကို သိရှိနိုင်ရန် tool schemas ပံ့ပိုးခြင်း
- `approval_mode` ဖြင့် tool ထမ်းဆောင်မှုကို ထိန်းချုပ်ခြင်း
- Pydantic models နှင့် `response_format` မှတဆင့် **ဖွဲ့စည်းထားသော output** ပြန်လည်ပေးသွင်းခြင်း

အဆိုပါ ဇာတ်ကောင်မှာ နေရာ duသွားမွေးရာ၊ ရရှိနိုင်မှုစစ်ဆေးခြင်းနှင့် စစ်တယ်ပျံသတင်းအချက်အလက် ရယူနိုင်သော **ခရီးသွားbooking ကိုယ်စားလှယ်** ဖြစ်သည်။


## သတ်မှတ်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ဒက်ကော်ရေတာနဲ့ ကိရိယာတွေ သတ်မှတ်ခြင်း

`@tool` ဒက်ကော်ရေတာက ပေါ့သာ Python ဖန်ရှင်ကို Agent တစ်ယောက်ခေါ်ယူနိုင်တဲ့ ကိရိယာအဖြစ်ပြောင်းလဲပေးပါတယ်။
အဓိကအချက်များ

- **docstring** က မော်ဒယ်ရဲ့ မြင်တွေ့နိုင်တဲ့ ကိရိယာ ဖော်ပြချက် ဖြစ်လာသည်။
- **Type annotations** (ဖော်ပြချက်ပါသော `Annotated` အပါအဝင်)က ကိရိယာ schema ကို သတ်မှတ်သည်။
- `approval_mode` က အသုံးပြုသူအနေနဲ့ ခေါ်ယူမှုတိုင်းကို အတည်ပြုရမယ့်အခြေအနေကိုထိန်းချုပ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## တူညီသောကိရိယာများဖြင့် Agent တစ်ခုဖန်တီးခြင်း

အသုံးပြုသူ၏မေးခွန်းကိုဖြေဆိုရန်လိုအပ်သောကိရိယာကိုမည်သည့်အခါမဆို တုံ့ပြန်နိုင်ရန်အတွက် ကိရိယာသုံးခုလုံးအား client ထံပေးပို့ပါ။


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ကိရိယာများဖြင့် ဖွဲ့စည်းထားသော အထွက်

`response_format` ကို Pydantic မော်ဒယ်တစ်ခုအဖြစ် သတ်မှတ်ခြင်းဖြင့်၊ အေးဂျင့်သည် အခမဲ့စာသားတစ်ခုပြန်ပေးပို့ခြင်းမဟုတ်ပဲ ကောင်းစွာအမျိုးအစားခွဲထားသော JSON အရာဝတ္ထုကို ပြန်သွားပေးရန် မလိုအပ်ပါ။ ၎င်းသည် အောက်ပိုင်းကုဒ်သည် ရလဒ်ကို အစီအစဉ်အလိုက် သုံးစွဲရန် လိုအပ်သောအခါ အသုံးဝင်သည်။


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ကိရိယာ အတည်ပြုခြင်း ပုံစံများ

`@tool` ပေါ်ရှိ `approval_mode` ပါရာမီတာက ကိရိယာကို ခေါ်ယူမှုတွေ ပြုလုပ်မယ့်အခါ လူသားတစ်ယောက်က အတည်ပြုခြင်း လိုအပ်မလိုကို ထိန်းချုပ်ပေးပါတယ်။

| ပုံစံ | အပြုအမူ |
|---|---|
| `"never_require"` | ကိရိယာကို မလိုအပ်ဘဲ အလိုအلكာ လည်ပတ်စေသည်။ |
| `"always_require"` | ကိရိယာကို ခေါ်ယူမဲ့ အခါတိုင်း သုံးစွဲသူမှ အတည်ပြုရမည်။ |

ဖက်စွဲသက်ရောက်မှုရှိတဲ့ ကိရိယာများအတွက် (ဥပမာ - လေယာဉ်လက်မှတ်ကောက်ခြင်း၊ ခရက်ဒစ်ကတ် စနစ်လေးဖြင့် နှိုင်းယှဉ်ကြေးဆောင်ခြင်း) `"always_require"` ကို အသုံးပြုပါက လူသားကို အတည်ပြုခြင်းအတွင်းမှာ ထားရှိနိုင်စေသည်။


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## အနှစ်ချုပ်

ဤသင်ခန်းစာတွင် သင်သည် အောက်ပါအကြောင်းများကို သင်ယူခဲ့သည်။

1. **@tool** decorator ကို အသုံးပြု၍ typed parameters နှင့် tool schema အဖြစ် အသုံးပြုနိုင်သော docstrings ရှိသည့် tools များကို သတ်မှတ်ခြင်း။
2. ဆက်တိုက်ခက်ခဲသော မေးခွန်းများကို ဖြေရှင်းနိုင်ရန်အတွက် agent သည် tools များစွာကို စုပေါင်းခေါ်ဆိုနိုင်ရန်။
3. `response_format` အဖြစ် Pydantic မော်ဒယ်တစ်ခုကို ပေးပို့၍ ဖွဲ့စည်းပုံရှိသော output ကို ပြန်ပိုင်းရန်။
4. သက်ဆိုင်ရာ လုပ်ငန်းစဉ်များတွင် လူကို ပါဝင်အတည်ပြုမှုရှိစေရန် `approval_mode` ဖြင့် tool အတည်ပြုမှုကို ထိန်းချုပ်ခြင်း။

ဤပုံစံများသည် အပြင်ဘက်စနစ်များနှင့် ဘေးလုံခြုံစွာ ဆက်သွယ်နိုင်သည့် ယုံကြည်စိတ်ချရပြီး ထုတ်လုပ်နိုင်သော agent များ ဖန်တီးခြင်းအတွက် အခြေခံဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
